# Performance Metrics for Machine Learning
## TTK 4260: Multivariate Data Analysis and ML

---

This interactive notebook accompanies the lecture on Performance Metrics. Work through each section, run the code cells, and experiment with the examples to build intuition about when and how to use different metrics.

### Table of Contents
1. [Why Metrics Matter](#1.-Why-Metrics-Matter)
2. [Regression Metrics](#2.-Regression-Metrics)
3. [Classification Metrics](#3.-Classification-Metrics)
4. [Probabilistic Predictions](#4.-Probabilistic-Predictions)
5. [Computer Vision Metrics](#5.-Computer-Vision-Metrics)
6. [Perceptual & Semantic Losses](#6.-Perceptual-&-Semantic-Losses)
7. [Distribution Distances](#7.-Distribution-Distances)
8. [Metric Selection Guide](#8.-Metric-Selection-Guide)

---

In [ ]:
# First, let's install and import all necessary libraries
# Run this cell first!

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    log_loss, brier_score_loss
)
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

print("✅ All libraries imported successfully!")

---
## 1. Why Metrics Matter

### The Cancer Detection Paradox 🏥

**Scenario:** A hospital builds a cancer detection model that achieves **99% accuracy**. Should they deploy it?

Let's investigate...

In [ ]:
# The Cancer Detection Paradox
# Dataset: 99% healthy, 1% cancer

np.random.seed(42)

# Create an imbalanced dataset
n_samples = 10000
n_cancer = 100  # 1% of the population
n_healthy = 9900  # 99% of the population

# True labels: 0 = Healthy, 1 = Cancer
y_true = np.array([1] * n_cancer + [0] * n_healthy)

# "Dumb" model: Always predicts healthy (0)
y_pred_dumb = np.zeros(n_samples)

# Calculate accuracy
accuracy_dumb = accuracy_score(y_true, y_pred_dumb)

print("=" * 50)
print("THE CANCER DETECTION PARADOX")
print("=" * 50)
print(f"\nDataset composition:")
print(f"  • Healthy patients: {n_healthy} (99%)")
print(f"  • Cancer patients:  {n_cancer} (1%)")
print(f"\n'Dumb' Model Strategy: Always predict 'Healthy'")
print(f"\n📊 Model Accuracy: {accuracy_dumb:.1%}")
print(f"\n🚨 But wait... how many cancer cases did we catch?")
print(f"   Cancer cases detected: {int(np.sum((y_true == 1) & (y_pred_dumb == 1)))} out of {n_cancer}")
print(f"\n⚠️  KEY MESSAGE: A 'terrible' model can look brilliant with the wrong metric!")

In [ ]:
# Visualize the paradox
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart of dataset composition
ax1 = axes[0]
sizes = [99, 1]
labels = ['Healthy (99%)', 'Cancer (1%)']
colors = ['#90EE90', '#FF6B6B']
explode = (0, 0.1)
ax1.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.0f%%',
        shadow=True, startangle=90)
ax1.set_title('Dataset Composition', fontsize=14, fontweight='bold')

# Bar chart showing the problem
ax2 = axes[1]
metrics = ['Accuracy', 'Cancer\nDetection Rate']
values = [99, 0]
colors = ['#4CAF50', '#F44336']
bars = ax2.bar(metrics, values, color=colors, edgecolor='black', linewidth=2)
ax2.set_ylim(0, 110)
ax2.set_ylabel('Percentage (%)', fontsize=12)
ax2.set_title('The "Dumb" Model Performance', fontsize=14, fontweight='bold')
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
             f'{val}%', ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('cancer_paradox.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 LESSON: The choice of metric can make a terrible model look brilliant!")

---
## 2. Regression Metrics

Regression tasks involve predicting continuous values. Common metrics include:

| Metric | Formula | Properties |
|--------|---------|------------|
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Robust to outliers, linear penalty |
| **MSE** | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ | Penalizes large errors, smooth gradient |
| **RMSE** | $\sqrt{MSE}$ | Same units as target variable |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proportion of variance explained |

In [ ]:
# Let's implement these metrics from scratch to understand them better

def mae(y_true, y_pred):
    """Mean Absolute Error"""
    return np.mean(np.abs(y_true - y_pred))

def mse(y_true, y_pred):
    """Mean Squared Error"""
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    """Root Mean Squared Error"""
    return np.sqrt(mse(y_true, y_pred))

def r_squared(y_true, y_pred):
    """Coefficient of Determination (R²)"""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

# Example data
y_actual = np.array([10, 12, 11, 13, 10, 15])
y_predicted = np.array([11, 11, 12, 12, 11, 14])

print("Regression Metrics Example")
print("=" * 40)
print(f"Actual values:    {y_actual}")
print(f"Predicted values: {y_predicted}")
print(f"\nMetrics:")
print(f"  MAE:  {mae(y_actual, y_predicted):.4f}")
print(f"  MSE:  {mse(y_actual, y_predicted):.4f}")
print(f"  RMSE: {rmse(y_actual, y_predicted):.4f}")
print(f"  R²:   {r_squared(y_actual, y_predicted):.4f}")

### ⚠️ TRAP #1: The Outlier Ambush

MSE is extremely sensitive to outliers because it squares the errors. A single outlier can dominate the entire metric!

In [ ]:
# TRAP #1: The Outlier Ambush

# Data with one outlier
actual = np.array([10, 12, 11, 13, 10, 100])  # Last value is an outlier!
predicted = np.array([11, 11, 12, 12, 11, 11])
errors = actual - predicted

print("=" * 60)
print("TRAP #1: THE OUTLIER AMBUSH")
print("=" * 60)

# Create a nice table
df = pd.DataFrame({
    'Actual': actual,
    'Predicted': predicted,
    'Error': errors,
    'Squared Error': errors**2
})
print("\nData with one outlier:")
print(df.to_string(index=True))

print("\n📊 Impact Analysis:")
print(f"  First 5 squared errors: {np.sum(errors[:5]**2)} (total)")
print(f"  Last error (outlier):   {errors[5]**2} (single point!)")
print(f"\n  Outlier contributes {errors[5]**2 / np.sum(errors**2) * 100:.1f}% of total squared error!")

print("\n📈 Metric Comparison:")
print(f"  MAE:  {mae(actual, predicted):.2f}")
print(f"  MSE:  {mse(actual, predicted):.2f}")
print(f"  RMSE: {rmse(actual, predicted):.2f}")
print("\n⚠️  ONE outlier dominates MSE completely!")

In [ ]:
# Visualize MAE vs MSE loss functions

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

errors_range = np.linspace(-5, 5, 100)

# MAE
ax1 = axes[0]
ax1.plot(errors_range, np.abs(errors_range), 'b-', linewidth=3, label='MAE')
ax1.set_xlabel('Error', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Mean Absolute Error (MAE)', fontsize=14, fontweight='bold')
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-5, 5)
ax1.set_ylim(0, 6)

# MSE
ax2 = axes[1]
ax2.plot(errors_range, errors_range**2, 'orange', linewidth=3, label='MSE')
ax2.set_xlabel('Error', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('Mean Squared Error (MSE)', fontsize=14, fontweight='bold')
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-5, 5)
ax2.set_ylim(0, 26)

# Add annotations
ax1.annotate('Linear growth', xy=(3, 3), fontsize=11, color='blue')
ax2.annotate('Quadratic growth\n(punishes outliers!)', xy=(2, 15), fontsize=11, color='darkorange')

plt.tight_layout()
plt.savefig('mae_vs_mse.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 USE MAE when: All errors matter equally, outliers should not dominate")
print("📌 USE MSE when: Large errors are catastrophic, need smooth gradients at 0")

### ⚠️ TRAP #2: R² Can Be Negative!

**Common misconception:** R² ∈ [0, 1]

**Reality:** R² ∈ (-∞, 1]

R² < 0 means your model is **worse than just predicting the mean**!

In [ ]:
# TRAP #2: R² Can Be Negative!

# Example: Perfectly inverted predictions
actual = np.array([1, 2, 3, 4, 5])
predicted_inverted = np.array([5, 4, 3, 2, 1])  # Completely wrong direction!

print("=" * 60)
print("TRAP #2: R² CAN BE NEGATIVE!")
print("=" * 60)
print("\nExample: Perfectly inverted predictions")
print(f"Actual:    {actual}")
print(f"Predicted: {predicted_inverted}")

# Calculate R²
y_mean = np.mean(actual)
ss_res = np.sum((actual - predicted_inverted) ** 2)
ss_tot = np.sum((actual - y_mean) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"\n📐 Calculation:")
print(f"  ȳ (mean) = {y_mean}")
print(f"  SS_res = Σ(y - ŷ)² = {ss_res}")
print(f"  SS_tot = Σ(y - ȳ)² = {ss_tot}")
print(f"  R² = 1 - {ss_res}/{ss_tot} = {r2}")

print(f"\n🚨 R² = {r2:.0f}")
print("\n💡 Interpretation: R² < 0 means the model is WORSE than predicting the mean!")

In [ ]:
# Visualize what negative R² means

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

actual = np.array([1, 2, 3, 4, 5])

# Perfect model
ax1 = axes[0]
predicted_perfect = actual.copy()
ax1.scatter(actual, actual, s=100, c='blue', label='Actual', zorder=3)
ax1.plot([0, 6], [0, 6], 'g--', linewidth=2, label='Perfect fit')
ax1.set_xlabel('Actual')
ax1.set_ylabel('Predicted')
ax1.set_title(f'Perfect Model\nR² = {r2_score(actual, predicted_perfect):.2f}', fontweight='bold')
ax1.legend()
ax1.set_xlim(0, 6)
ax1.set_ylim(0, 6)

# Mean prediction
ax2 = axes[1]
predicted_mean = np.full_like(actual, np.mean(actual), dtype=float)
ax2.scatter(actual, predicted_mean, s=100, c='orange', label='Predictions', zorder=3)
ax2.axhline(y=np.mean(actual), color='orange', linestyle='--', linewidth=2, label='Mean line')
ax2.plot([0, 6], [0, 6], 'g--', linewidth=2, alpha=0.3)
ax2.set_xlabel('Actual')
ax2.set_ylabel('Predicted')
ax2.set_title(f'Always Predict Mean\nR² = {r2_score(actual, predicted_mean):.2f}', fontweight='bold')
ax2.legend()
ax2.set_xlim(0, 6)
ax2.set_ylim(0, 6)

# Inverted model
ax3 = axes[2]
predicted_inverted = np.array([5, 4, 3, 2, 1])
ax3.scatter(actual, predicted_inverted, s=100, c='red', label='Predictions', zorder=3)
ax3.plot([0, 6], [6, 0], 'r--', linewidth=2, label='Inverted fit')
ax3.plot([0, 6], [0, 6], 'g--', linewidth=2, alpha=0.3)
ax3.set_xlabel('Actual')
ax3.set_ylabel('Predicted')
ax3.set_title(f'Inverted Model\nR² = {r2_score(actual, predicted_inverted):.2f}', fontweight='bold', color='red')
ax3.legend()
ax3.set_xlim(0, 6)
ax3.set_ylim(0, 6)

plt.tight_layout()
plt.savefig('r2_negative.png', dpi=150, bbox_inches='tight')
plt.show()

### Huber Loss: Best of Both Worlds

Huber loss combines the advantages of MAE and MSE:
- **Quadratic** near zero (smooth gradient like MSE)
- **Linear** far from zero (robust to outliers like MAE)

$$L_\delta(a) = \begin{cases} \frac{1}{2}a^2 & \text{if } |a| \leq \delta \\ \delta(|a| - \frac{1}{2}\delta) & \text{otherwise} \end{cases}$$

In [ ]:
# Huber Loss Implementation and Visualization

def huber_loss(error, delta=1.0):
    """Huber loss function"""
    abs_error = np.abs(error)
    quadratic = 0.5 * error ** 2
    linear = delta * (abs_error - 0.5 * delta)
    return np.where(abs_error <= delta, quadratic, linear)

# Visualize all three loss functions
errors = np.linspace(-5, 5, 200)
delta = 1.5

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(errors, np.abs(errors), 'b--', linewidth=2, alpha=0.7, label='MAE')
ax.plot(errors, errors**2, 'orange', linestyle='--', linewidth=2, alpha=0.7, label='MSE')
ax.plot(errors, huber_loss(errors, delta), 'g-', linewidth=3, label=f'Huber (δ={delta})')

# Mark the transition points
ax.axvline(x=-delta, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=delta, color='gray', linestyle=':', alpha=0.5)
ax.fill_between([-delta, delta], [0, 0], [10, 10], alpha=0.1, color='green', label='Quadratic region')

ax.set_xlabel('Error', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Huber Loss: Best of Both Worlds', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(-5, 5)
ax.set_ylim(0, 10)
ax.grid(True, alpha=0.3)

# Add annotations
ax.annotate('Quadratic\n(smooth gradient)', xy=(0, 0.5), fontsize=10, ha='center')
ax.annotate('Linear\n(robust)', xy=(3.5, 4), fontsize=10, ha='center')

plt.tight_layout()
plt.savefig('huber_loss.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 Huber Loss is ideal when you want:")
print("   • Smooth gradients near zero for stable optimization")
print("   • Robustness to outliers in your data")

### 🧪 Interactive Exercise: Regression Metrics

Try changing the outlier value and see how it affects different metrics!

In [ ]:
# Interactive Exercise: Explore how outliers affect metrics

def compare_metrics_with_outlier(outlier_value):
    """Compare regression metrics with different outlier values"""
    # Normal data
    actual_normal = np.array([10, 12, 11, 13, 10])
    pred_normal = np.array([11, 11, 12, 12, 11])
    
    # Data with outlier
    actual_outlier = np.append(actual_normal, outlier_value)
    pred_outlier = np.append(pred_normal, 11)
    
    print(f"\n{'='*60}")
    print(f"Outlier value: {outlier_value}")
    print(f"{'='*60}")
    print(f"\n{'Metric':<15} {'Without Outlier':<20} {'With Outlier':<20}")
    print("-" * 55)
    print(f"{'MAE':<15} {mae(actual_normal, pred_normal):<20.4f} {mae(actual_outlier, pred_outlier):<20.4f}")
    print(f"{'MSE':<15} {mse(actual_normal, pred_normal):<20.4f} {mse(actual_outlier, pred_outlier):<20.4f}")
    print(f"{'RMSE':<15} {rmse(actual_normal, pred_normal):<20.4f} {rmse(actual_outlier, pred_outlier):<20.4f}")
    
    # Calculate increase factors
    mae_increase = mae(actual_outlier, pred_outlier) / mae(actual_normal, pred_normal)
    mse_increase = mse(actual_outlier, pred_outlier) / mse(actual_normal, pred_normal)
    
    print(f"\n📈 Increase factors:")
    print(f"   MAE increased by: {mae_increase:.1f}x")
    print(f"   MSE increased by: {mse_increase:.1f}x")

# Try different outlier values
print("🔬 EXPERIMENT: How do outliers affect regression metrics?")
for outlier in [15, 50, 100, 500]:
    compare_metrics_with_outlier(outlier)

---
## 3. Classification Metrics

### The Confusion Matrix

The confusion matrix is the foundation for understanding classification metrics.

|  | **Predicted Positive** | **Predicted Negative** |
|--|------------------------|------------------------|
| **Actually Positive** | TP (True Positive) | FN (False Negative) |
| **Actually Negative** | FP (False Positive) | TN (True Negative) |

In [ ]:
# Understanding the Confusion Matrix

def plot_confusion_matrix_explained():
    """Create an annotated confusion matrix diagram"""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create a simple confusion matrix
    cm_data = np.array([[0.4, 0.15], [0.1, 0.35]])  # Normalized values for coloring
    
    # Plot
    im = ax.imshow(cm_data, cmap='RdYlGn', vmin=0, vmax=0.5)
    
    # Labels
    labels = [['TP\n(True Positive)\n\n✅ Correctly identified\nas positive', 
               'FP\n(False Positive)\n\n❌ Type I Error\n"False Alarm"'],
              ['FN\n(False Negative)\n\n❌ Type II Error\n"Miss"', 
               'TN\n(True Negative)\n\n✅ Correctly identified\nas negative']]
    
    colors = [['darkgreen', 'darkred'], ['darkred', 'darkgreen']]
    
    for i in range(2):
        for j in range(2):
            text = ax.text(j, i, labels[i][j], ha='center', va='center', 
                          fontsize=10, color='white', fontweight='bold')
    
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Predicted\nPositive', 'Predicted\nNegative'], fontsize=12)
    ax.set_yticklabels(['Actually\nPositive', 'Actually\nNegative'], fontsize=12)
    ax.set_title('The Confusion Matrix Explained', fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.savefig('confusion_matrix_explained.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_confusion_matrix_explained()

In [ ]:
# Key Classification Metrics

def calculate_classification_metrics(y_true, y_pred):
    """Calculate and explain all key classification metrics"""
    # Get confusion matrix values
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Calculate metrics
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'Specificity': specificity,
        'F1': f1
    }

# Example
np.random.seed(42)
y_true = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1])
y_pred = np.array([1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1])

metrics = calculate_classification_metrics(y_true, y_pred)

print("CLASSIFICATION METRICS EXPLAINED")
print("=" * 60)
print(f"\nConfusion Matrix Values:")
print(f"  TP = {metrics['TP']}, TN = {metrics['TN']}, FP = {metrics['FP']}, FN = {metrics['FN']}")

print("\n" + "-" * 60)
print(f"\n📊 ACCURACY = (TP + TN) / (TP + TN + FP + FN)")
print(f"   = ({metrics['TP']} + {metrics['TN']}) / ({metrics['TP']} + {metrics['TN']} + {metrics['FP']} + {metrics['FN']})")
print(f"   = {metrics['Accuracy']:.2%}")
print(f"   📌 'What fraction of ALL predictions were correct?'")

print(f"\n📊 PRECISION = TP / (TP + FP)")
print(f"   = {metrics['TP']} / ({metrics['TP']} + {metrics['FP']})")
print(f"   = {metrics['Precision']:.2%}")
print(f"   📌 'When I say YES, am I right?'")

print(f"\n📊 RECALL (Sensitivity) = TP / (TP + FN)")
print(f"   = {metrics['TP']} / ({metrics['TP']} + {metrics['FN']})")
print(f"   = {metrics['Recall']:.2%}")
print(f"   📌 'Did I find all the actual positives?'")

print(f"\n📊 SPECIFICITY = TN / (TN + FP)")
print(f"   = {metrics['TN']} / ({metrics['TN']} + {metrics['FP']})")
print(f"   = {metrics['Specificity']:.2%}")
print(f"   📌 'Did I correctly identify the negatives?'")

print(f"\n📊 F1 SCORE = 2 × (Precision × Recall) / (Precision + Recall)")
print(f"   = 2 × ({metrics['Precision']:.2f} × {metrics['Recall']:.2f}) / ({metrics['Precision']:.2f} + {metrics['Recall']:.2f})")
print(f"   = {metrics['F1']:.2%}")
print(f"   📌 'Harmonic mean of Precision and Recall'")

### ⚠️ TRAP #3: The Accuracy Paradox

With imbalanced data, accuracy can be **completely misleading**!

In [ ]:
# TRAP #3: The Accuracy Paradox

print("=" * 60)
print("TRAP #3: THE ACCURACY PARADOX")
print("=" * 60)
print("\nScenario: Fraud detection (1% fraud rate)")

# Create imbalanced dataset
n_total = 10000
n_fraud = 100  # 1%
n_normal = 9900  # 99%

y_true = np.array([1] * n_fraud + [0] * n_normal)

# "Dumb" Model: Always predict "normal" (no fraud)
y_pred_dumb = np.zeros(n_total)

# "Smart" Model: Catches 10% of fraud with some false positives
y_pred_smart = np.zeros(n_total)
y_pred_smart[:10] = 1  # Catches 10 fraud cases
y_pred_smart[n_fraud:n_fraud+10] = 1  # 10 false positives

print("\n" + "─" * 60)
print("📊 COMPARING TWO MODELS:")
print("─" * 60)

for name, y_pred in [('"Dumb" Model (always predicts Normal)', y_pred_dumb),
                      ('"Smart" Model (tries to detect fraud)', y_pred_smart)]:
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    print(f"\n{name}")
    print(f"  Accuracy:  {acc:.1%}")
    print(f"  Precision: {prec:.1%}")
    print(f"  Recall:    {rec:.1%}")
    print(f"  F1 Score:  {f1:.1%}")
    print(f"  Fraud caught: {int(np.sum((y_true == 1) & (y_pred == 1)))}/{n_fraud}")

print("\n" + "=" * 60)
print("⚠️  BOTH HAVE ~99% ACCURACY!")
print("   But the 'dumb' model catches ZERO fraud cases!")
print("   In imbalanced data, accuracy is nearly MEANINGLESS!")
print("=" * 60)

In [ ]:
# Visualize the Accuracy Paradox

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model comparison
ax1 = axes[0]
models = ['"Dumb" Model', '"Smart" Model']
metrics_names = ['Accuracy', 'Recall\n(Fraud Detection)']

x = np.arange(len(models))
width = 0.35

dumb_values = [99.0, 0.0]
smart_values = [99.0, 10.0]

bars1 = ax1.bar(x - width/2, dumb_values, width, label='Accuracy', color='#4CAF50', edgecolor='black')
bars2 = ax1.bar(x + width/2, [0, 10], width, label='Recall', color='#2196F3', edgecolor='black')

# Add 'dumb' model recall bar (0)
ax1.bar(0 + width/2, 0.5, width, color='#F44336', edgecolor='black')  # Red bar showing 0

ax1.set_ylabel('Percentage (%)', fontsize=12)
ax1.set_title('The Accuracy Paradox', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(models, fontsize=11)
ax1.legend()
ax1.set_ylim(0, 110)

# Add value labels
for bar, val in zip(bars1, dumb_values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{val:.0f}%', ha='center', fontweight='bold')
ax1.text(0 + width/2, 5, '0%', ha='center', fontweight='bold', color='white')
ax1.text(1 + width/2, 12, '10%', ha='center', fontweight='bold')

# What metric to use?
ax2 = axes[1]
metrics_comparison = ['Accuracy', 'Recall', 'F1 Score', 'PR-AUC']
good_for_imbalanced = ['❌ Misleading', '✅ Shows reality', '✅ Balanced view', '✅ Best choice']
colors = ['#F44336', '#4CAF50', '#4CAF50', '#4CAF50']

ax2.barh(metrics_comparison, [99, 10, 15, 25], color=colors, edgecolor='black')
ax2.set_xlabel('"Smart" Model Score', fontsize=12)
ax2.set_title('Which Metric Reveals the Truth?', fontsize=14, fontweight='bold')
ax2.set_xlim(0, 110)

for i, (metric, status) in enumerate(zip(metrics_comparison, good_for_imbalanced)):
    ax2.text(102, i, status, va='center', fontsize=10)

plt.tight_layout()
plt.savefig('accuracy_paradox.png', dpi=150, bbox_inches='tight')
plt.show()

### ⚠️ TRAP #4: The Precision-Recall Trade-off

By adjusting the classification threshold, you can trade precision for recall (and vice versa). You cannot maximize both!

In [ ]:
# TRAP #4: Precision-Recall Trade-off

# Generate a more realistic dataset
np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           n_redundant=5, n_classes=2, weights=[0.7, 0.3],
                           random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train a logistic regression model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# Get probability predictions
y_proba = model.predict_proba(X_test)[:, 1]

# Calculate precision-recall for different thresholds
thresholds = np.arange(0.1, 0.95, 0.05)
precisions = []
recalls = []

for thresh in thresholds:
    y_pred = (y_proba >= thresh).astype(int)
    precisions.append(precision_score(y_test, y_pred, zero_division=0))
    recalls.append(recall_score(y_test, y_pred, zero_division=0))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Threshold effect
ax1 = axes[0]
ax1.plot(thresholds, precisions, 'b-o', linewidth=2, markersize=6, label='Precision')
ax1.plot(thresholds, recalls, 'r-s', linewidth=2, markersize=6, label='Recall')
ax1.set_xlabel('Classification Threshold', fontsize=12)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Precision vs Recall at Different Thresholds', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1.05)

# Add annotations
ax1.annotate('High threshold\n(Conservative)', xy=(0.85, 0.9), fontsize=10,
             bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax1.annotate('Low threshold\n(Aggressive)', xy=(0.15, 0.3), fontsize=10,
             bbox=dict(boxstyle='round', facecolor='lightyellow'))

# PR Curve
ax2 = axes[1]
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_proba)
ax2.plot(recall_curve, precision_curve, 'purple', linewidth=3)
ax2.fill_between(recall_curve, precision_curve, alpha=0.3, color='purple')
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title(f'Precision-Recall Curve\n(AP = {average_precision_score(y_test, y_proba):.3f})', 
              fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('precision_recall_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 KEY INSIGHT: Same model, different thresholds!")
print("   Moving the threshold traces the PR curve.")
print("   You CANNOT have both high Precision AND high Recall.")

### ⚠️ TRAP #5: F1 Score Hides Critical Information

Two models with the same F1 score can have very different precision/recall profiles!

In [ ]:
# TRAP #5: F1 Score Hides Critical Information

print("=" * 60)
print("TRAP #5: F1 SCORE HIDES CRITICAL INFORMATION")
print("=" * 60)
print("\nScenario: Cancer screening (Positive = Has Cancer)")

# Two models with similar F1 but very different P/R profiles
models = [
    {'name': 'Model A', 'precision': 0.90, 'recall': 0.10},
    {'name': 'Model B', 'precision': 0.45, 'recall': 0.45},
]

print("\n" + "─" * 60)
print(f"{'Model':<12} {'Precision':<12} {'Recall':<12} {'F1 Score':<12}")
print("─" * 60)

for m in models:
    f1 = 2 * m['precision'] * m['recall'] / (m['precision'] + m['recall'])
    m['f1'] = f1
    print(f"{m['name']:<12} {m['precision']:<12.0%} {m['recall']:<12.0%} {f1:<12.0%}")

print("\n" + "─" * 60)
print("\n🔍 ANALYSIS:")
print("\n  Model A:")
print("    • High precision (90%) - When it says 'cancer', it's usually right")
print("    • BUT catches only 10% of cancers!")
print("    • 90% of cancer patients are MISSED!")
print("\n  Model B:")
print("    • Lower precision (45%) - More false alarms")
print("    • BUT catches 45% of cancers")
print("    • Catches 4.5× more cancers than Model A!")

print("\n" + "=" * 60)
print("⚠️  FOR CANCER SCREENING: Model B is much better!")
print("   Missing cancer (FN) is far worse than a false alarm (FP).")
print("   F1 alone hides WHICH errors you're making!")
print("=" * 60)

In [ ]:
# Visualize the F1 trap

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar comparison
ax1 = axes[0]
x = np.arange(2)
width = 0.25

prec_vals = [0.90, 0.45]
rec_vals = [0.10, 0.45]
f1_vals = [0.18, 0.45]

ax1.bar(x - width, prec_vals, width, label='Precision', color='#2196F3', edgecolor='black')
ax1.bar(x, rec_vals, width, label='Recall', color='#FF9800', edgecolor='black')
ax1.bar(x + width, f1_vals, width, label='F1', color='#9C27B0', edgecolor='black')

ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('F1 Hides the Precision-Recall Balance', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(['Model A', 'Model B'], fontsize=11)
ax1.legend()
ax1.set_ylim(0, 1)

# Visual of what this means for cancer detection
ax2 = axes[1]

# Show cancer patients caught
n_cancer = 100

model_a_caught = int(n_cancer * 0.10)
model_b_caught = int(n_cancer * 0.45)

ax2.barh(['Model A', 'Model B'], [model_a_caught, model_b_caught], 
         color=['#F44336', '#4CAF50'], edgecolor='black', height=0.5)
ax2.barh(['Model A', 'Model B'], [n_cancer - model_a_caught, n_cancer - model_b_caught],
         left=[model_a_caught, model_b_caught], color=['#FFCDD2', '#C8E6C9'], 
         edgecolor='black', height=0.5)

ax2.set_xlabel('Number of Cancer Patients', fontsize=12)
ax2.set_title('Cancer Cases Caught (out of 100)', fontsize=14, fontweight='bold')
ax2.set_xlim(0, 110)

ax2.text(model_a_caught/2, 0, f'{model_a_caught}\ncaught', ha='center', va='center', fontweight='bold', color='white')
ax2.text(model_a_caught + (100-model_a_caught)/2, 0, f'{100-model_a_caught}\nmissed', ha='center', va='center')
ax2.text(model_b_caught/2, 1, f'{model_b_caught}\ncaught', ha='center', va='center', fontweight='bold', color='white')
ax2.text(model_b_caught + (100-model_b_caught)/2, 1, f'{100-model_b_caught}\nmissed', ha='center', va='center')

plt.tight_layout()
plt.savefig('f1_trap.png', dpi=150, bbox_inches='tight')
plt.show()

### ROC Curve and AUC

The **ROC (Receiver Operating Characteristic)** curve plots:
- **X-axis:** False Positive Rate (FPR) = FP / (FP + TN)
- **Y-axis:** True Positive Rate (TPR) = TP / (TP + FN) = Recall

**AUC (Area Under Curve):** Probability that a random positive is ranked higher than a random negative.

In [ ]:
# ROC Curve Demonstration

# Use our earlier model
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)

fig, ax = plt.subplots(figsize=(10, 8))

# ROC curve
ax.plot(fpr, tpr, 'b-', linewidth=3, label=f'Model (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random (AUC = 0.5)', alpha=0.7)

# Fill area under curve
ax.fill_between(fpr, tpr, alpha=0.3, color='blue')

# Perfect classifier point
ax.plot(0, 1, 'go', markersize=15, label='Perfect classifier')

# Mark a specific threshold
idx = np.argmin(np.abs(thresholds - 0.5))
ax.plot(fpr[idx], tpr[idx], 'ro', markersize=12, label=f'Threshold = 0.5')
ax.annotate(f'FPR={fpr[idx]:.2f}\nTPR={tpr[idx]:.2f}', 
            xy=(fpr[idx], tpr[idx]), xytext=(fpr[idx]+0.1, tpr[idx]-0.1),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('True Positive Rate (TPR)', fontsize=12)
ax.set_title('ROC Curve Explained', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

# Add interpretation guide
ax.text(0.6, 0.3, 'Better →', fontsize=12, color='blue')
ax.annotate('', xy=(0.2, 0.8), xytext=(0.5, 0.5),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))

plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 ROC-AUC Interpretation:")
print("   • AUC = 1.0: Perfect classifier")
print("   • AUC = 0.5: Random guessing")
print("   • AUC < 0.5: Worse than random (predictions are inverted!)")

### ⚠️ TRAP #6: ROC-AUC Can Mislead with Imbalanced Data

ROC-AUC uses **rates**, not counts. In imbalanced data, a small FPR can still mean many false positives!

In [ ]:
# TRAP #6: ROC-AUC with Imbalanced Data

print("=" * 60)
print("TRAP #6: ROC-AUC CAN MISLEAD WITH IMBALANCED DATA")
print("=" * 60)
print("\nScenario: Rare disease (0.1% prevalence)")
print("          10 sick per 10,000 people")

n_total = 10000
n_sick = 10
n_healthy = 9990

print("\n📊 Model with AUC = 0.95 (sounds great!)")
print("\nAt threshold giving 80% TPR (catches 8 of 10 sick):")

# If TPR = 80%, we catch 8 sick people
tp = 8
fn = 2

# From a typical ROC curve with AUC=0.95, at 80% TPR, FPR might be ~5%
fpr_example = 0.05
fp = int(fpr_example * n_healthy)  # 5% of 9990 = ~500

print(f"   FPR from ROC curve: {fpr_example:.0%}")
print(f"   But 5% of {n_healthy} healthy people = {fp} FALSE ALARMS!")

precision = tp / (tp + fp)
print(f"\n   📈 Precision = {tp} / ({tp} + {fp}) = {precision:.1%}")
print(f"   Only 1 in {int(1/precision)} positive predictions is correct!")

print("\n" + "─" * 60)
print("\n💡 WHY DOES THIS HAPPEN?")
print("   ROC uses RATES, not COUNTS.")
print("   5% of a huge number (9,990) is still huge (500)!")
print("\n   SOLUTION: Use PR-AUC for imbalanced/rare events!")

In [ ]:
# Compare ROC-AUC vs PR-AUC on imbalanced data

# Create a very imbalanced dataset
np.random.seed(42)
X_imb, y_imb = make_classification(n_samples=10000, n_features=20, n_informative=10,
                                    n_classes=2, weights=[0.99, 0.01],  # 1% positive class
                                    random_state=42)

X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb)

# Train model
model_imb = LogisticRegression(random_state=42, max_iter=1000)
model_imb.fit(X_train_imb, y_train_imb)
y_proba_imb = model_imb.predict_proba(X_test_imb)[:, 1]

# Calculate both AUCs
roc_auc_imb = roc_auc_score(y_test_imb, y_proba_imb)
pr_auc_imb = average_precision_score(y_test_imb, y_proba_imb)

print(f"\nDataset: {np.sum(y_test_imb == 1)} positives, {np.sum(y_test_imb == 0)} negatives")
print(f"\nROC-AUC: {roc_auc_imb:.3f} (looks great!)")
print(f"PR-AUC:  {pr_auc_imb:.3f} (reveals the challenge)")

# Plot both curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
ax1 = axes[0]
fpr_imb, tpr_imb, _ = roc_curve(y_test_imb, y_proba_imb)
ax1.plot(fpr_imb, tpr_imb, 'b-', linewidth=3, label=f'ROC (AUC = {roc_auc_imb:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.5)
ax1.fill_between(fpr_imb, tpr_imb, alpha=0.3)
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curve (Looks Good!)', fontsize=14, fontweight='bold', color='green')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# PR Curve
ax2 = axes[1]
prec_imb, rec_imb, _ = precision_recall_curve(y_test_imb, y_proba_imb)
ax2.plot(rec_imb, prec_imb, 'purple', linewidth=3, label=f'PR (AUC = {pr_auc_imb:.3f})')
ax2.axhline(y=np.mean(y_test_imb), color='k', linestyle='--', linewidth=2, alpha=0.5,
            label=f'Baseline = {np.mean(y_test_imb):.3f}')
ax2.fill_between(rec_imb, prec_imb, alpha=0.3, color='purple')
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('PR Curve (Shows Reality!)', fontsize=14, fontweight='bold', color='purple')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('roc_vs_pr.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 RULE OF THUMB:")
print("   • ROC-AUC: Use for balanced data")
print("   • PR-AUC: Use for imbalanced data / rare events")

---
## 4. Probabilistic Predictions

When models output probabilities (not just class labels), we need metrics that evaluate the **quality of those probabilities**.

In [ ]:
# Why Hard Predictions Aren't Enough

print("=" * 60)
print("WHY PROBABILITY QUALITY MATTERS")
print("=" * 60)

print("\nScenario: Two models predict 'spam' correctly.")
print("\n" + "─" * 60)
print(f"{'Model':<10} {'Confidence':<15} {'Correct?':<12} {'Log Loss':<12}")
print("─" * 60)

# Both correct, but different confidences
y_true_ex = [1]  # Actually spam

model_a_prob = 0.51
model_b_prob = 0.99

log_loss_a = -np.log(model_a_prob)
log_loss_b = -np.log(model_b_prob)

print(f"{'Model A':<10} {model_a_prob:<15.0%} {'✓':<12} {log_loss_a:<12.4f}")
print(f"{'Model B':<10} {model_b_prob:<15.0%} {'✓':<12} {log_loss_b:<12.4f}")

print("\n💡 Both models are CORRECT by accuracy!")
print("   But Model B's prediction is FAR more useful.")
print(f"\n   Model B's Log Loss is {log_loss_a/log_loss_b:.0f}× better.")

In [ ]:
# Log Loss and Brier Score

def calculate_log_loss_manual(y_true, y_prob):
    """Manual log loss calculation"""
    epsilon = 1e-15  # Prevent log(0)
    y_prob = np.clip(y_prob, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

def calculate_brier_score_manual(y_true, y_prob):
    """Manual Brier score calculation"""
    return np.mean((y_prob - y_true) ** 2)

# Example with multiple predictions
np.random.seed(42)
y_true_prob = np.array([1, 1, 0, 0, 1, 0, 1, 0, 1, 1])

# Good model - confident and correct
y_prob_good = np.array([0.9, 0.85, 0.1, 0.2, 0.95, 0.15, 0.8, 0.1, 0.9, 0.75])

# Bad model - uncertain or wrong
y_prob_bad = np.array([0.6, 0.4, 0.5, 0.5, 0.55, 0.45, 0.5, 0.5, 0.6, 0.55])

# Overconfident wrong model
y_prob_overconfident = np.array([0.95, 0.95, 0.05, 0.95, 0.95, 0.95, 0.95, 0.05, 0.95, 0.95])

print("COMPARING PROBABILISTIC METRICS")
print("=" * 60)
print(f"\nTrue labels: {y_true_prob}")
print("\n" + "─" * 60)
print(f"{'Model':<25} {'Log Loss':<15} {'Brier Score':<15} {'Accuracy':<10}")
print("─" * 60)

for name, probs in [('Good (confident & correct)', y_prob_good),
                     ('Bad (uncertain)', y_prob_bad),
                     ('Overconfident & wrong', y_prob_overconfident)]:
    ll = log_loss(y_true_prob, probs)
    bs = brier_score_loss(y_true_prob, probs)
    acc = accuracy_score(y_true_prob, (probs >= 0.5).astype(int))
    print(f"{name:<25} {ll:<15.4f} {bs:<15.4f} {acc:<10.0%}")

print("\n💡 KEY INSIGHTS:")
print("   • Log Loss SEVERELY punishes confident wrong predictions")
print("   • Brier Score is like MSE for probabilities")
print("   • The 'overconfident wrong' model is punished the most!")

In [ ]:
# Visualize Log Loss behavior

fig, ax = plt.subplots(figsize=(10, 6))

probs = np.linspace(0.001, 0.999, 100)

# Log loss when true label is 1
loss_true_1 = -np.log(probs)

# Log loss when true label is 0
loss_true_0 = -np.log(1 - probs)

ax.plot(probs, loss_true_1, 'b-', linewidth=3, label='True label = 1 (should predict high)')
ax.plot(probs, loss_true_0, 'r-', linewidth=3, label='True label = 0 (should predict low)')

ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Log Loss', fontsize=12)
ax.set_title('Log Loss: Punishes Confident Wrong Predictions', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 5)

# Add annotations
ax.annotate('Predicting 0.01 when\ntrue label is 1:\nLOSS ≈ 4.6!', 
            xy=(0.01, 4.6), xytext=(0.2, 4),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='blue'))

ax.annotate('Predicting 0.99 when\ntrue label is 1:\nLOSS ≈ 0.01', 
            xy=(0.99, 0.01), xytext=(0.7, 1),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='blue'))

plt.tight_layout()
plt.savefig('log_loss.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Computer Vision Metrics

### Signal-to-Noise Ratio (SNR) and Peak SNR (PSNR)

PSNR is commonly used for image quality assessment:

$$PSNR = 10 \cdot \log_{10}\left(\frac{MAX^2}{MSE}\right)$$

For 8-bit images (MAX = 255): $PSNR \approx 48.13 - 10\log_{10}(MSE)$

In [ ]:
# PSNR Implementation

def calculate_psnr(original, distorted, max_val=255):
    """Calculate Peak Signal-to-Noise Ratio"""
    mse_val = np.mean((original.astype(float) - distorted.astype(float)) ** 2)
    if mse_val == 0:
        return float('inf')
    return 10 * np.log10((max_val ** 2) / mse_val)

# Create a simple test image
np.random.seed(42)
original = np.random.randint(0, 256, (100, 100)).astype(np.uint8)

# Different types of distortions with same MSE
noise_level = 30

# 1. Gaussian noise
noisy = np.clip(original.astype(float) + np.random.normal(0, noise_level, original.shape), 0, 255).astype(np.uint8)

# 2. Blurring (simple box filter)
from scipy.ndimage import uniform_filter
blurred = uniform_filter(original, size=5).astype(np.uint8)

# 3. Shift by 1 pixel
shifted = np.roll(original, 1, axis=1)

print("=" * 60)
print("PSNR: SAME VALUE, DIFFERENT PERCEPTUAL QUALITY")
print("=" * 60)

for name, img in [('Noisy', noisy), ('Blurred', blurred), ('Shifted (1px)', shifted)]:
    psnr = calculate_psnr(original, img)
    mse_val = np.mean((original.astype(float) - img.astype(float)) ** 2)
    print(f"\n{name}:")
    print(f"  MSE:  {mse_val:.2f}")
    print(f"  PSNR: {psnr:.2f} dB")

print("\n" + "=" * 60)
print("⚠️  TRAP #7: Same PSNR can look VERY different!")
print("   PSNR measures mathematical similarity, NOT perceptual similarity!")
print("=" * 60)

In [ ]:
# Visualize the PSNR trap

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Create a more visually interesting test image (gradient with pattern)
x = np.linspace(0, 4*np.pi, 100)
y = np.linspace(0, 4*np.pi, 100)
X, Y = np.meshgrid(x, y)
original = ((np.sin(X) * np.sin(Y) + 1) * 127).astype(np.uint8)

# Distortions
noisy = np.clip(original.astype(float) + np.random.normal(0, 25, original.shape), 0, 255).astype(np.uint8)
blurred = uniform_filter(original, size=7).astype(np.uint8)
shifted = np.roll(original, 3, axis=1)

images = [original, blurred, noisy, shifted]
titles = ['Original', 
          f'Blurred\nPSNR={calculate_psnr(original, blurred):.1f}dB',
          f'Noisy\nPSNR={calculate_psnr(original, noisy):.1f}dB',
          f'Shifted\nPSNR={calculate_psnr(original, shifted):.1f}dB']

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle('PSNR Trap: Same Metric, Different Visual Quality', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('psnr_trap.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 LESSON: Use SSIM or learned perceptual metrics (LPIPS) for human-aligned quality!")

### Structural Similarity Index (SSIM)

SSIM considers three components:
1. **Luminance** - Mean intensity comparison
2. **Contrast** - Variance comparison  
3. **Structure** - Covariance (pattern similarity)

$$SSIM(x,y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)}$$

In [ ]:
# SSIM Implementation (simplified)

def calculate_ssim_simple(img1, img2, C1=6.5025, C2=58.5225):
    """Simplified SSIM calculation (global, not windowed)"""
    img1 = img1.astype(float)
    img2 = img2.astype(float)
    
    mu1 = np.mean(img1)
    mu2 = np.mean(img2)
    
    sigma1_sq = np.var(img1)
    sigma2_sq = np.var(img2)
    sigma12 = np.cov(img1.flatten(), img2.flatten())[0, 1]
    
    ssim = ((2 * mu1 * mu2 + C1) * (2 * sigma12 + C2)) / \
           ((mu1**2 + mu2**2 + C1) * (sigma1_sq + sigma2_sq + C2))
    
    return ssim

# Compare PSNR and SSIM on different distortions
print("COMPARING PSNR VS SSIM")
print("=" * 60)
print(f"\n{'Distortion':<20} {'PSNR (dB)':<15} {'SSIM':<15}")
print("─" * 60)

for name, img in [('None (identical)', original),
                   ('Noisy', noisy), 
                   ('Blurred', blurred), 
                   ('Shifted', shifted)]:
    psnr = calculate_psnr(original, img)
    ssim = calculate_ssim_simple(original, img)
    print(f"{name:<20} {psnr:<15.2f} {ssim:<15.4f}")

print("\n💡 SSIM better captures perceptual differences!")
print("   • Noise destroys structure → low SSIM")
print("   • Blur smooths but preserves some structure → medium SSIM")
print("   • Shift preserves all structure → high SSIM")

### Intersection over Union (IoU) / Jaccard Index

IoU is the standard metric for object detection and segmentation:

$$IoU = \frac{Area_{intersection}}{Area_{union}} = \frac{TP}{TP + FP + FN}$$

In [ ]:
# IoU Demonstration

def calculate_iou(box1, box2):
    """Calculate IoU between two bounding boxes [x1, y1, x2, y2]"""
    # Calculate intersection
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    if x2 < x1 or y2 < y1:
        return 0.0
    
    intersection = (x2 - x1) * (y2 - y1)
    
    # Calculate areas
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

# Visualize different IoU scenarios
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Ground truth box
gt_box = [20, 20, 80, 80]

# Different prediction scenarios
scenarios = [
    ([20, 20, 80, 80], 'Perfect Match'),
    ([30, 30, 90, 90], 'Good Overlap'),
    ([50, 50, 110, 110], 'Partial Overlap'),
    ([100, 100, 150, 150], 'No Overlap'),
]

for ax, (pred_box, title) in zip(axes, scenarios):
    iou = calculate_iou(gt_box, pred_box)
    
    # Draw boxes
    ax.set_xlim(0, 160)
    ax.set_ylim(160, 0)  # Invert y-axis for image coordinates
    
    # Ground truth (green)
    gt_rect = plt.Rectangle((gt_box[0], gt_box[1]), gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
                            fill=False, edgecolor='green', linewidth=3, linestyle='-', label='Ground Truth')
    ax.add_patch(gt_rect)
    
    # Prediction (red)
    pred_rect = plt.Rectangle((pred_box[0], pred_box[1]), pred_box[2]-pred_box[0], pred_box[3]-pred_box[1],
                              fill=False, edgecolor='red', linewidth=3, linestyle='--', label='Prediction')
    ax.add_patch(pred_rect)
    
    ax.set_title(f'{title}\nIoU = {iou:.2f}', fontsize=11, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

# Add legend to first plot
axes[0].legend(loc='upper right', fontsize=9)

plt.suptitle('Intersection over Union (IoU) Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('iou_examples.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Common IoU Thresholds:")
print("   • IoU ≥ 0.5: COCO 'loose' threshold")
print("   • IoU ≥ 0.75: COCO 'strict' threshold")
print("   • mAP@0.5:0.95: Average over multiple thresholds")

---
## 6. Perceptual & Semantic Losses

### Why Pixel-wise Losses Fail for Image Generation

MSE in image generation produces **blurry** results because when the model is uncertain about the correct pixel value, minimizing MSE leads to **averaging**.

In [ ]:
# Why MSE causes blur

print("=" * 60)
print("WHY MSE CAUSES BLUR IN IMAGE GENERATION")
print("=" * 60)

print("\nScenario: Model is uncertain if edge is at position 10 or 12")
print("\nOption 1: Sharp edge at position 10")
print("   Pixels: [0, 0, 0, 0, 0, 255, 255, 255, 255, 255]")
print("\nOption 2: Sharp edge at position 12")
print("   Pixels: [0, 0, 0, 0, 0, 0, 0, 255, 255, 255]")

# If true edge is at 10 but model averages:
sharp_10 = np.array([0, 0, 0, 0, 0, 255, 255, 255, 255, 255])
sharp_12 = np.array([0, 0, 0, 0, 0, 0, 0, 255, 255, 255])
averaged = (sharp_10 + sharp_12) / 2

print("\n📊 MSE-optimal prediction (average):")
print(f"   Pixels: {averaged.astype(int)}")
print("\n⚠️  The 'average' creates BLUR at the edge!")
print("   MSE rewards playing it safe, not being sharp.")

In [ ]:
# Visualize MSE blur effect

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Create 2D versions
sharp = np.zeros((50, 100))
sharp[:, 50:] = 255

blurry = np.zeros((50, 100))
for i, val in enumerate(np.linspace(0, 255, 20)):
    blurry[:, 40+i:41+i] = val
blurry[:, 60:] = 255

ax1 = axes[0]
ax1.imshow(sharp, cmap='gray', vmin=0, vmax=255)
ax1.set_title('Sharp Edge\n(Ground Truth)', fontsize=12, fontweight='bold')
ax1.axis('off')

ax2 = axes[1]
ax2.imshow(blurry, cmap='gray', vmin=0, vmax=255)
ax2.set_title('MSE-Optimal Prediction\n(Blurry!)', fontsize=12, fontweight='bold')
ax2.axis('off')

# Show 1D cross-section
ax3 = axes[2]
x = np.arange(100)
ax3.plot(x, sharp[25, :], 'g-', linewidth=3, label='Sharp (Ground Truth)')
ax3.plot(x, blurry[25, :], 'r--', linewidth=3, label='Blurry (MSE-optimal)')
ax3.set_xlabel('Pixel Position', fontsize=11)
ax3.set_ylabel('Intensity', fontsize=11)
ax3.set_title('Cross-Section at Row 25', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mse_blur.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 SOLUTION: Use Perceptual Loss (feature-space comparison)")
print("   or Adversarial Loss (GAN) to encourage sharp outputs!")

---
## 7. Distribution Distances

When comparing probability distributions (e.g., in generative models), we need special metrics.

In [ ]:
# KL Divergence

def kl_divergence(p, q):
    """Calculate KL(P||Q) - KL divergence from Q to P"""
    # Add small epsilon to avoid log(0)
    epsilon = 1e-10
    p = np.clip(p, epsilon, 1)
    q = np.clip(q, epsilon, 1)
    return np.sum(p * np.log(p / q))

def js_divergence(p, q):
    """Calculate Jensen-Shannon divergence"""
    m = (p + q) / 2
    return 0.5 * kl_divergence(p, m) + 0.5 * kl_divergence(q, m)

# Example distributions
P = np.array([0.9, 0.1])
Q = np.array([0.5, 0.5])

print("=" * 60)
print("KL DIVERGENCE: NOT SYMMETRIC!")
print("=" * 60)
print(f"\nP = {P}")
print(f"Q = {Q}")

kl_pq = kl_divergence(P, Q)
kl_qp = kl_divergence(Q, P)
js = js_divergence(P, Q)

print(f"\n📊 KL(P||Q) = {kl_pq:.4f}")
print(f"📊 KL(Q||P) = {kl_qp:.4f}")
print(f"\n⚠️  TRAP #11: KL(P||Q) ≠ KL(Q||P)!")
print(f"   Difference: {abs(kl_pq - kl_qp):.4f}")

print(f"\n📊 Jensen-Shannon Divergence = {js:.4f}")
print("   JS is symmetric and bounded [0, 1]!")

In [ ]:
# KL Divergence can be INFINITE!

print("\n" + "=" * 60)
print("TRAP #12: KL DIVERGENCE CAN BE INFINITE!")
print("=" * 60)

P_risky = np.array([0.5, 0.5, 0.0])  # P has zero probability for outcome 3
Q_risky = np.array([0.33, 0.33, 0.34])  # Q assigns probability to all outcomes

print(f"\nP = {P_risky}  (zero probability for outcome 3)")
print(f"Q = {Q_risky}  (non-zero for all outcomes)")

# KL(Q||P) involves log(Q/P) where P=0
print(f"\n📊 KL(P||Q) = {kl_divergence(P_risky, Q_risky):.4f}  (finite)")
print(f"📊 KL(Q||P) = INFINITE! (Q assigns probability where P doesn't)")

print("\n💡 This happens when Q(x) > 0 but P(x) = 0")
print("   The log(Q/P) term becomes log(Q/0) = ∞")

In [ ]:
# Visualize KL divergence directions

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x = np.linspace(-4, 6, 200)

# True distribution P (narrow)
P_dist = 1/(np.sqrt(2*np.pi)*0.8) * np.exp(-0.5*((x-1)/0.8)**2)

# Approximation Q
Q_narrow = 1/(np.sqrt(2*np.pi)*0.8) * np.exp(-0.5*((x-1)/0.8)**2)  # Same as P
Q_wide = 1/(np.sqrt(2*np.pi)*2) * np.exp(-0.5*((x-1)/2)**2)  # Wider
Q_shifted = 1/(np.sqrt(2*np.pi)*0.8) * np.exp(-0.5*((x-3)/0.8)**2)  # Shifted

# Plot 1: Mode-covering (Forward KL)
ax1 = axes[0]
ax1.fill_between(x, P_dist, alpha=0.5, color='blue', label='P (target)')
ax1.plot(x, Q_wide, 'r--', linewidth=2, label='Q (wider)')
ax1.set_title('Forward KL: KL(P||Q)\n"Mode-covering"', fontsize=11, fontweight='bold')
ax1.set_xlabel('x')
ax1.set_ylabel('Density')
ax1.legend()
ax1.text(0.5, 0.95, 'Q spreads to cover all of P', transform=ax1.transAxes, fontsize=9,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat'))

# Plot 2: Mode-seeking (Reverse KL)
ax2 = axes[1]
ax2.fill_between(x, P_dist, alpha=0.5, color='blue', label='P (target)')
ax2.plot(x, Q_narrow, 'r--', linewidth=2, label='Q (matches mode)')
ax2.set_title('Reverse KL: KL(Q||P)\n"Mode-seeking"', fontsize=11, fontweight='bold')
ax2.set_xlabel('x')
ax2.set_ylabel('Density')
ax2.legend()
ax2.text(0.5, 0.95, 'Q focuses on high-density regions of P', transform=ax2.transAxes, fontsize=9,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat'))

# Plot 3: Non-overlapping (KL = infinity)
ax3 = axes[2]
ax3.fill_between(x, P_dist, alpha=0.5, color='blue', label='P (target)')
ax3.fill_between(x, Q_shifted, alpha=0.5, color='red', label='Q (shifted)')
ax3.set_title('Non-overlapping\nKL = ∞, Wasserstein finite!', fontsize=11, fontweight='bold')
ax3.set_xlabel('x')
ax3.set_ylabel('Density')
ax3.legend()

plt.tight_layout()
plt.savefig('kl_divergence.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 When to use which direction:")
print("   • KL(P||Q): Forward - use when you want Q to cover all of P (mode-covering)")
print("   • KL(Q||P): Reverse - use when you want Q to match P's modes (VAEs)")
print("   • Wasserstein: When distributions don't overlap (GANs)")

---
## 8. Metric Selection Guide

### The 6-Question Framework

In [ ]:
# Metric Selection Guide

print("="*70)
print("METRIC SELECTION: THE 6-QUESTION FRAMEWORK")
print("="*70)

questions = [
    ("1. What's the decision?", "Match metric to task type"),
    ("2. Cost of FP vs FN?", "Precision vs Recall, cost matrix"),
    ("3. Is data imbalanced?", "MCC, balanced accuracy, PR-AUC"),
    ("4. Output probabilities?", "Log loss, Brier, ECE"),
    ("5. Comparing images?", "IoU, Dice, SSIM, mAP"),
    ("6. Predicting distributions?", "KL, JS, Wasserstein"),
]

for question, answer in questions:
    print(f"\n  📋 {question}")
    print(f"     → {answer}")

In [ ]:
# Quick Reference Table

print("\n" + "="*80)
print("QUICK REFERENCE: METRICS BY TASK")
print("="*80)

reference = [
    ("Balanced Classification", "Accuracy, F1", "—"),
    ("Imbalanced Classification", "PR-AUC, MCC, F1", "Accuracy, ROC-AUC"),
    ("Regression with Outliers", "MAE, Huber", "MSE"),
    ("Regression without Outliers", "MSE, RMSE, R²", "—"),
    ("Probabilistic Classification", "Log Loss, Brier, ECE", "Hard accuracy"),
    ("Segmentation", "IoU, Dice, Hausdorff", "Pixel accuracy"),
    ("Object Detection", "mAP@various IoU", "Single threshold"),
    ("Image Quality", "SSIM, LPIPS", "PSNR alone"),
    ("Image Generation", "FID, LPIPS, perceptual", "MSE, PSNR"),
    ("Ranking/Retrieval", "NDCG, MAP, MRR", "Raw accuracy"),
    ("Distribution Matching", "Wasserstein, JS, KL", "MSE on parameters"),
]

print(f"\n{'Task':<30} {'Primary Metrics':<25} {'Avoid':<20}")
print("-" * 80)

for task, primary, avoid in reference:
    print(f"{task:<30} {primary:<25} {avoid:<20}")

### ⚠️ Goodhart's Law: A Final Warning

> *"When a measure becomes a target, it ceases to be a good measure."*

In [ ]:
# Goodhart's Law Examples

print("="*70)
print("GOODHART'S LAW: WHEN METRICS BECOME TARGETS")
print("="*70)

examples = [
    ("Click-through rate", "Clickbait"),
    ("Accuracy", "Ignoring minority classes"),
    ("BLEU score", "Grammatically correct but meaningless text"),
    ("Response time", "Incomplete answers"),
    ("Lines of code", "Bloated, unreadable code"),
    ("Publication count", "Salami-slicing research"),
]

print("\n🎯 Real-world examples of metric gaming:")
print("-" * 70)

for metric, consequence in examples:
    print(f"  • Optimizing {metric:<20} → {consequence}")

print("\n" + "="*70)
print("💡 TAKEAWAYS:")
print("   1. Use MULTIPLE metrics")
print("   2. Understand what you're TRULY optimizing")
print("   3. ALWAYS visualize your results")
print("   4. Consider the BUSINESS/DOMAIN implications")
print("="*70)

---
## 🧪 Final Exercises

Test your understanding with these exercises!

In [ ]:
# Exercise 1: Which metric would you use?

print("="*70)
print("EXERCISE 1: CHOOSE THE RIGHT METRIC")
print("="*70)

scenarios = [
    "1. Detecting fraudulent credit card transactions (0.1% fraud rate)",
    "2. Predicting house prices with some luxury outliers",
    "3. Medical diagnosis where missing a disease is much worse than a false alarm",
    "4. Evaluating image super-resolution quality",
    "5. Comparing output of a language model to reference text",
]

print("\nFor each scenario, think about which metric(s) would be most appropriate:")
for s in scenarios:
    print(f"\n  {s}")
    print("  Your answer: _____________________")

print("\n" + "-"*70)
print("Scroll down for suggested answers...")
print("\n" * 5)

print("SUGGESTED ANSWERS:")
print("-"*70)
answers = [
    "PR-AUC, Recall, F1 (NOT accuracy or ROC-AUC - data is too imbalanced)",
    "MAE or Huber Loss (NOT MSE - outliers will dominate)",
    "High Recall is critical; use F2-score or set low threshold",
    "SSIM, LPIPS (NOT just PSNR - doesn't capture perceptual quality)",
    "BLEU, ROUGE, but also perplexity and human evaluation",
]

for i, (s, a) in enumerate(zip(scenarios, answers), 1):
    print(f"\n{i}. {a}")

In [ ]:
# Exercise 2: Calculate metrics by hand

print("="*70)
print("EXERCISE 2: CALCULATE METRICS BY HAND")
print("="*70)

print("\nGiven this confusion matrix for a cancer detector:")
print("\n          Predicted")
print("           +    -")
print("Actual + [ 80   20 ]  (100 cancer patients)")
print("       - [ 50  850 ]  (900 healthy patients)")

print("\nCalculate:")
print("  a) Accuracy")
print("  b) Precision")
print("  c) Recall (Sensitivity)")
print("  d) Specificity")
print("  e) F1 Score")

print("\n" + "-"*70)
print("Your calculations:")
print("\n" * 3)

# Answers
TP, FN = 80, 20
FP, TN = 50, 850

print("\nANSWERS:")
print("-"*70)
print(f"a) Accuracy = (TP+TN)/(TP+TN+FP+FN) = ({TP}+{TN})/({TP}+{TN}+{FP}+{FN}) = {(TP+TN)/(TP+TN+FP+FN):.1%}")
print(f"b) Precision = TP/(TP+FP) = {TP}/({TP}+{FP}) = {TP/(TP+FP):.1%}")
print(f"c) Recall = TP/(TP+FN) = {TP}/({TP}+{FN}) = {TP/(TP+FN):.1%}")
print(f"d) Specificity = TN/(TN+FP) = {TN}/({TN}+{FP}) = {TN/(TN+FP):.1%}")
prec = TP/(TP+FP)
rec = TP/(TP+FN)
f1 = 2*prec*rec/(prec+rec)
print(f"e) F1 = 2×P×R/(P+R) = 2×{prec:.2f}×{rec:.2f}/({prec:.2f}+{rec:.2f}) = {f1:.1%}")

---
## Summary: Key Traps to Remember

### Regression
- ⚠️ Outliers dominate MSE
- ⚠️ R² can be negative
- ⚠️ Same metric, different trustworthiness

### Classification
- ⚠️ Accuracy paradox (imbalance)
- ⚠️ Precision-Recall trade-off
- ⚠️ F1 hides information
- ⚠️ ROC-AUC misleads with rare events

### Computer Vision
- ⚠️ PSNR ≠ perceptual quality
- ⚠️ IoU misses boundary shifts
- ⚠️ mAP@0.5 vs mAP@0.75 discrepancy

### Advanced
- ⚠️ KL is not symmetric
- ⚠️ KL = ∞ with non-overlapping support
- ⚠️ Easy negatives = no learning

---

**Remember:** The choice of metric fundamentally shapes what your model learns. Choose wisely!

---
*End of notebook*